# Cell 1 — Leakage control and evaluation splits

**Purpose:** preserve original source splits, establish defensible leakage constraints and
reserve separate research training, development-validation, calibration, internal-test and
optional external-test partitions for the civilian thermal UAV search-and-rescue project.
This notebook freezes a protocol only when all configured requirements are met.

**Required inputs:** a successful Notebook 02 audit with published, hashed manifests and
matching mounted images; a matching real-data Notebook 03 bundle containing its exploration
exposure record; project source/configuration. Visibility labels are optional and remain
masked where absent. External datasets require separate verified audit bundles and paths.

**Outputs:** a unique run containing original split provenance, a separate research assignment
index when feasible, duplicate/group/exposure reports, fixed subgroup rules, and a content
fingerprint. Missing inputs produce a blocked report and a null split fingerprint. Infeasible
proposals are explicitly NOT FROZEN. This is a protocol/data smoke stage, not training.

No final-test label counts, size/visibility statistics, metrics or operating thresholds guide
allocation. Assignment uses source roles, available verified grouping, duplicate constraints,
the central seed and grouped image counts. Unknown physical groups stay unknown. Dataset bytes
and manifest geometry are checked for integrity without interpreting held-out labels for tuning.
Every notebook runs independently from a fresh kernel. Stop after Notebook 04.

In [118]:
import os
from pathlib import Path

experiments = Path("/kaggle/working/artifacts/experiments")

os.environ["THERMAL_PROJECT_ROOT"] = (
    "/kaggle/input/datasets/dule11/thermal-project-root"
)
os.environ["THERMAL_DATASET_ROOT"] = (
    "/kaggle/input/datasets/pandrii000/"
    "hituav-a-highaltitude-infrared-thermal-dataset/hit-uav"
)
os.environ["THERMAL_AUDIT_RUN"] = str(
    experiments / "thermal-sar-dataset-audit_20261120T024854579539Z_30fc5c422d_af17e401"
)
os.environ["THERMAL_EXPLORATION_RUN"] = str(
    experiments / "thermal-sar-exploration_20261120T030644840893Z_31979a8a20_f3caf1cb"
)

for key in ("THERMAL_AUDIT_RUN", "THERMAL_EXPLORATION_RUN"):
    run = Path(os.environ[key])
    if not (run / "run.json").is_file():
        raise FileNotFoundError(run / "run.json")
    print(f"{key} = {run}")

THERMAL_AUDIT_RUN = /kaggle/working/artifacts/experiments/thermal-sar-dataset-audit_20261120T024854579539Z_30fc5c422d_af17e401
THERMAL_EXPLORATION_RUN = /kaggle/working/artifacts/experiments/thermal-sar-exploration_20261120T030644840893Z_31979a8a20_f3caf1cb


In [119]:
# Cell 2 — Verify the Notebook 02 audit, Notebook 03 export, and HIT-UAV mount.
import json
import logging
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("/kaggle/input/datasets/dule11/thermal-project-root")
DATASET_ROOT = Path(
    "/kaggle/input/datasets/pandrii000/"
    "hituav-a-highaltitude-infrared-thermal-dataset/hit-uav"
)
EXPERIMENTS_ROOT = Path("/kaggle/working/artifacts/experiments")
AUDIT_RUN = EXPERIMENTS_ROOT / (
    "thermal-sar-dataset-audit_20261120T024854579539Z_30fc5c422d_af17e401"
)
EXPLORATION_RUN = EXPERIMENTS_ROOT / (
    "thermal-sar-exploration_20261120T030644840893Z_31979a8a20_f3caf1cb"
)

required_files = (
    PROJECT_ROOT / "configs/base.yaml",
    PROJECT_ROOT / "src/data/split_artifacts.py",
    DATASET_ROOT / "dataset.yaml",
    AUDIT_RUN / "run.json",
    AUDIT_RUN / "logs/audit_report.json",
    EXPLORATION_RUN / "run.json",
    EXPLORATION_RUN / "logs/exploration_report.json",
)
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing Notebook 04 inputs:\n" + "\n".join(missing))

SETTINGS_PATH_HINT = os.environ.get("THERMAL_SPLIT_SETTINGS")
NEAR_REVIEW_HINT = os.environ.get("THERMAL_NEAR_DUPLICATE_REVIEWS")
OUTPUT_ROOT_HINT = os.environ.get("THERMAL_OUTPUT_ROOT")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.split_artifacts import (
    construct_plan,
    export_splits,
    fixed_strata,
    initialize_splits,
    partition_balance,
    prepare_constraints,
    verify_frozen_splits,
)
from src.data.splits import GROUP_FIELDS, validate_assignments

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

context = initialize_splits(
    PROJECT_ROOT,
    audit_run=AUDIT_RUN,
    exploration_run=EXPLORATION_RUN,
    dataset_root=DATASET_ROOT,
    settings_path=Path(SETTINGS_PATH_HINT).expanduser() if SETTINGS_PATH_HINT else None,
    near_review_path=Path(NEAR_REVIEW_HINT).expanduser() if NEAR_REVIEW_HINT else None,
    output_root=Path(OUTPUT_ROOT_HINT).expanduser() if OUTPUT_ROOT_HINT else None,
)
run_dir = Path(context["run_dir"])

print("Data ready:", context["data_ready"])
print("Remaining blockers:", json.dumps(context["blockers"], indent=2))
print("Prior audit:", context["audit"]["folder"])
print("Prior exploration:", context["exploration"]["folder"])
print("Notebook 04 run directory:", run_dir)

Data ready: True
Remaining blockers: []
Prior audit: /kaggle/working/artifacts/experiments/thermal-sar-dataset-audit_20261120T024854579539Z_30fc5c422d_af17e401
Prior exploration: /kaggle/working/artifacts/experiments/thermal-sar-exploration_20261120T030644840893Z_31979a8a20_f3caf1cb
Notebook 04 run directory: /kaggle/working/artifacts/experiments/thermal-sar-splits_20261120T031439308936Z_2f74c32423_7d1c2c3d


In [120]:
# Cell 3 — Inspect verified source flight, sequence and location identifiers and their missingness.
for field in GROUP_FIELDS:
    available = sum(row.get(field) is not None for row in context["images"])
    print(field, "available:", available, "/", len(context["images"]))
print("Values require source metadata availability/provenance. Filenames and target sizes never create group IDs.")

flight_id available: 0 / 2866
sequence_id available: 0 / 2866
location_id available: 0 / 2866
Values require source metadata availability/provenance. Filenames and target sizes never create group IDs.


In [121]:
# Cell 4 — Build conservative connected components from known groups, exact duplicates and near candidates.
prepare_constraints(context)
if context["components"] is not None:
    print("Grouping evidence:", json.dumps({key: value for key, value in context["components"].items()
                                           if key not in {"groups", "by_image"}}, indent=2))
    print("Leakage components:", len(context["components"]["groups"]))
    print("Near-duplicate coverage:", context["near_duplicates"])
print("Unreviewed near candidates are conservative grouping constraints, not confirmed duplicate ground truth.")
print("A false-pair review can remove its candidate edge; it cannot break verified metadata or exact-hash links.")

Grouping evidence: {
  "coverage": {
    "flight_id": {
      "available": 0,
      "images": 2866
    },
    "sequence_id": {
      "available": 0,
      "images": 2866
    },
    "location_id": {
      "available": 0,
      "images": 2866
    }
  },
  "edge_counts": {
    "candidate_near_duplicate_constraint": 13
  },
  "unreviewed_candidates_grouped": 13,
  "reviewed_false_pairs": 0,
  "flight_disjointness": "not_established",
  "limitation": "Generated component IDs are leakage constraints, never fabricated flight IDs; unobserved grouping remains unknown"
}
Leakage components: 2853
Near-duplicate coverage: {'method': '64-bit display-space difference hash; manual review required', 'pairs_compared': 100000, 'total_pairs': 4105545, 'complete': False, 'hamming_threshold': 4}
Unreviewed near candidates are conservative grouping constraints, not confirmed duplicate ground truth.
A false-pair review can remove its candidate edge; it cannot break verified metadata or exact-hash links.


## Cell 5 — Internal and external evaluation protocols

The **official benchmark index** copies original split names/provenance unchanged and
discloses known overlap. The **research index** is separately named and fingerprinted.
Configured original train/validation sources form a development pool; whole leakage components
are allocated into training, development validation and held-out-from-training calibration.
This pooling is an additional research protocol, not an unchanged official benchmark run.

Internal test must come from explicitly declared original test sources. Notebook 03 exposure
records protect against calling already explored test images untouched. Conflicting components
are quarantined in full, and any resulting test subset/exclusions are disclosed. Development
EDA may precede calibration reservation; this history is counted honestly. No claim is made
that calibration labels were never viewed, only that the calibration partition is separate
from training/validation and not used to fit the detector.

External datasets are registered from their own verified audits. Their object labels are not
parsed for assignment, subgroup balancing or tuning; all retained external images are external
test only. No fine-tuning on external test is permitted under the zero-shot protocol. Exact
duplicate constraints operate across datasets, while source group IDs are namespaced by dataset.

Without complete verified flight identifiers, flight disjointness is not established. Generated
component IDs cannot fill that gap. Missing visibility labels stay masked. No inference of
identity ground truth or tracking performance is allowed from sequence filenames/continuity.

In [122]:
# Cell 6 — Review immutable original source-role declarations; retain official split provenance.
print(json.dumps(context["settings"]["source_role_map"], indent=2))
print("Original split samples:", [{"dataset_id": row["dataset_id"], "image_id": row["image_id"],
                                  "official_split": row["split"], "evidence": row["split_provenance"]}
                                 for row in context["images"][:5]])
print("Unknown source roles are quarantined, not silently converted into training or test.")

{
  "train": "development_pool",
  "training": "development_pool",
  "val": "development_pool",
  "validation": "development_pool",
  "test": "internal_test"
}
Original split samples: [{'dataset_id': 'HIT-UAV', 'image_id': 'images/test/0_100_30_0_03307.jpg', 'official_split': 'test', 'evidence': 'Kaggle mirror dataset.yaml and mounted test folders'}, {'dataset_id': 'HIT-UAV', 'image_id': 'images/test/0_100_30_0_08074.jpg', 'official_split': 'test', 'evidence': 'Kaggle mirror dataset.yaml and mounted test folders'}, {'dataset_id': 'HIT-UAV', 'image_id': 'images/test/0_100_30_0_08091.jpg', 'official_split': 'test', 'evidence': 'Kaggle mirror dataset.yaml and mounted test folders'}, {'dataset_id': 'HIT-UAV', 'image_id': 'images/test/0_100_30_0_08100.jpg', 'official_split': 'test', 'evidence': 'Kaggle mirror dataset.yaml and mounted test folders'}, {'dataset_id': 'HIT-UAV', 'image_id': 'images/test/0_100_40_0_03327.jpg', 'official_split': 'test', 'evidence': 'Kaggle mirror dataset.yaml and

In [123]:
# Cell 7 — Create a deterministic research proposal from whole components, where feasible.
construct_plan(context)
if context["plan"] is not None:
    print("Proposed partition image counts:", context["plan"]["partition_counts"])
    print("Proposal blockers:", context["plan"]["blockers"])
print("No label-based allocation, balancing seed search, or reroll after seeing test outcomes occurs.")

Proposed partition image counts: {'quarantine': 19, 'internal_test': 564, 'training': 1598, 'calibration': 342, 'development_validation': 343}
Proposal blockers: []
No label-based allocation, balancing seed search, or reroll after seeing test outcomes occurs.


In [124]:
# Cell 8 — Inspect training and development-validation partitions; image fractions are approximate under grouping.
print("Requested development fractions:", context["settings"]["development_fractions"])
if context["plan"] is not None:
    for name in ("training", "development_validation"):
        print(name, context["plan"]["partition_counts"].get(name, 0))
print("Every active partition needs a distinct component. Too few groups block the freeze.")

Requested development fractions: {'training': 0.7, 'development_validation': 0.15, 'calibration': 0.15}
training 1598
development_validation 343
Every active partition needs a distinct component. Too few groups block the freeze.


In [125]:
# Cell 9 — Reserve calibration separately from detector training and development validation.
if context["plan"] is not None:
    print("Calibration images:", context["plan"]["partition_counts"].get("calibration", 0))
    print("Calibration images already included in development EDA:", context["plan"]["calibration_prior_exploration_images"])
print("Disabling calibration requires a recorded reason. No calibration parameters are fitted here.")

Calibration images: 342
Calibration images already included in development EDA: 342
Disabling calibration requires a recorded reason. No calibration parameters are fitted here.


In [126]:
# Cell 10 — Reserve internal test only from declared untouched test-source components.
if context["plan"] is not None:
    print("Internal-test images:", context["plan"]["partition_counts"].get("internal_test", 0))
    print("Quarantined images:", context["plan"]["partition_counts"].get("quarantine", 0))
print("Test images with a known link to explored data or conflicting source roles are quarantined as a whole component.")
print("Absent eligible test data block the default protocol; test data are not manufactured from explored training images.")

Internal-test images: 564
Quarantined images: 19
Test images with a known link to explored data or conflicting source roles are quarantined as a whole component.
Absent eligible test data block the default protocol; test data are not manufactured from explored training images.


In [127]:
# Cell 11 — Register actual external-test audits; no target-label tuning or adaptation.
print(json.dumps(context["external_registry"], indent=2))
if not context["external_registry"]:
    print("External evaluation disabled: no external dataset audit/root was supplied.")
print("External object-file hashes are verified for integrity, but object/test labels are not parsed or summarized.")

[]
External evaluation disabled: no external dataset audit/root was supplied.
External object-file hashes are verified for integrity, but object/test labels are not parsed or summarized.


In [128]:
# Cell 12 — Check image, exact-hash, known-group and conservative-component overlap.
if context["plan"] is not None:
    validate_assignments(context["images"], context["plan"]["assignments"], context["components"])
    print(json.dumps(context["leakage_audit"], indent=2))
print("Research constraints and official-source overlap are separate findings. Official source rows remain unchanged.")

{
  "research_constraints_passed": true,
  "official_exact_cross_source_count": 0,
  "official_known_group_cross_source_count": 0,
  "official_exact_findings": [],
  "official_known_group_findings": [],
  "cross_source_conservative_components": 7,
  "official_index_modified": false,
  "limitation": "Conservative near-candidate components are not confirmed duplicates; official overlap is disclosed, not silently repaired"
}
Research constraints and official-source overlap are separate findings. Official source rows remain unchanged.


In [129]:
# Cell 13 — Report development size/visibility/metadata coverage without exposing final-test label distributions.
balance = partition_balance(context)
print(json.dumps(balance, indent=2))
print("Internal/external test label statistics are withheld until final evaluation. No image-count allocation is changed from this report.")

{
  "calibration": {
    "images": 342,
    "components": 342,
    "metadata_available": {
      "flight_id": 0,
      "sequence_id": 0,
      "location_id": 0
    },
    "person_objects_noncrowd": 1530,
    "size_counts": {
      "small": 1527,
      "medium": 3
    },
    "visibility_labels_available": 0
  },
  "development_validation": {
    "images": 343,
    "components": 343,
    "metadata_available": {
      "flight_id": 0,
      "sequence_id": 0,
      "location_id": 0
    },
    "person_objects_noncrowd": 1396,
    "size_counts": {
      "small": 1392,
      "medium": 4
    },
    "visibility_labels_available": 0
  },
  "internal_test": {
    "images": 564,
    "components": 563,
    "metadata_available": {
      "flight_id": 0,
      "sequence_id": 0,
      "location_id": 0
    },
    "label_statistics": "withheld_until_final_evaluation"
  },
  "quarantine": {
    "images": 19,
    "components": 7,
    "metadata_available": {
      "flight_id": 0,
      "sequence_id": 0,
    

In [130]:
# Cell 14 — Gate tracking partitions on actual identity ground truth; current adapters have none.
print("Verified sequence metadata images:", sum(row.get("sequence_id") is not None for row in context["images"]))
print("Tracking split/evaluation disabled: verified ground-truth person identities are unavailable in current audit adapters.")
print("Sequence IDs constrain detection splits, but do not substitute for tracking identities or prove usable video continuity.")

Verified sequence metadata images: 0
Tracking split/evaluation disabled: verified ground-truth person identities are unavailable in current audit adapters.
Sequence IDs constrain detection splits, but do not substitute for tracking identities or prove usable video continuity.


In [131]:
# Cell 15 — Freeze documented original-coordinate subgroup rules, not tuned test-set thresholds.
print(json.dumps(fixed_strata(context["settings"]), indent=2))
print("Unavailable visibility/altitude have explicit missingness policies. No target-size information becomes an inference-time input.")

{
  "schema_version": "1.0",
  "size": {
    "coordinate_space": "original_image_pixels",
    "measure": "bbox_area_px2",
    "boundaries": [
      1024,
      9216
    ],
    "intervals": "small < a; medium a <= A < b; large >= b"
  },
  "visibility": {
    "categories": [
      "fully_visible",
      "partially_occluded",
      "heavily_occluded"
    ],
    "missing_policy": "mask_unavailable_labels"
  },
  "altitude": {
    "units": "meters",
    "boundaries": [
      60,
      90,
      120
    ],
    "missing_policy": "separate_unavailable_group",
    "source": "verified_metadata_only"
  },
  "test_label_statistics_computed": false
}
Unavailable visibility/altitude have explicit missingness policies. No target-size information becomes an inference-time input.


In [132]:
# Cell 16 — Export immutable official provenance and freeze research membership only if every gate passes.
export_splits(context)
print("Artifacts:", run_dir)
print("Status:", context["status"], "research frozen:", context["frozen"])
print("Split fingerprint:", context["config"]["split"]["fingerprint"])
if context["frozen"]:
    verified = verify_frozen_splits(run_dir)
    print("Frozen content verified:", verified["identifier"], verified["fingerprint"])
else:
    print("No frozen research index/fingerprint was emitted. Any proposal is explicitly NOT_FROZEN.")

Artifacts: /kaggle/working/artifacts/experiments/thermal-sar-splits_20261120T031439308936Z_2f74c32423_7d1c2c3d
Status: blocked_split_constraints research frozen: False
Split fingerprint: None
No frozen research index/fingerprint was emitted. Any proposal is explicitly NOT_FROZEN.


In [133]:
# Cell 17 — Inspect saved balance/leakage reports and remaining limitations.
saved_report = json.loads((run_dir / "logs/split_report.json").read_text(encoding="utf-8"))
print("Remaining blockers:", saved_report["blockers"])
print("Split report:", run_dir / "logs/split_report.json")
print("Near-review template:", run_dir / "logs/near_duplicate_review_template.json")
print("No data deletion, training, installations, downloads, metrics or old-result overwrites were performed.")

Remaining blockers: ['Near-duplicate comparison coverage is incomplete; increase budget or document the explicit remaining limitation']
Split report: /kaggle/working/artifacts/experiments/thermal-sar-splits_20261120T031439308936Z_2f74c32423_7d1c2c3d/logs/split_report.json
Near-review template: /kaggle/working/artifacts/experiments/thermal-sar-splits_20261120T031439308936Z_2f74c32423_7d1c2c3d/logs/near_duplicate_review_template.json
No data deletion, training, installations, downloads, metrics or old-result overwrites were performed.


## Cell 18 — Freeze status, run instructions and limitations

A successful result is `split_protocol_frozen` with a non-null fingerprint verified by the
shared downstream loader. `blocked_missing_inputs` or `blocked_split_constraints` is an honest
incomplete data/protocol state; no frozen index should be used for training in that case.
Unchanged official splits may contain disclosed overlap. Research constraints prevent the
known linked components from crossing active partitions, but unknown physical grouping and
partial candidate search remain limitations. A partial near-duplicate audit requires an
explicit recorded reason; it never becomes an exhaustive audit or flight-disjoint guarantee.

**Kaggle/Colab:** provide project code, successful matching Notebook 02 and 03 bundles, and
actual mounted images. Set `PROJECT_ROOT_HINT`, `AUDIT_RUN_HINT`, `EXPLORATION_RUN_HINT` and
`DATASET_ROOT_HINT` in Cell 2. Copy `configs/splits.yaml` to writable working storage, verify
its source-role mapping and group/calibration policies from actual evidence, and set
`SETTINGS_PATH_HINT`. Restart and Run All. Kaggle outputs go under `/kaggle/working/artifacts/experiments/`;
Colab under `/content/artifacts/experiments/`. Optional external audits are listed explicitly
in that settings copy. No source mount name is assumed and no Drive mount/install runs.

If conservative candidate edges make the split infeasible, preserve the reports, review the
indicated pairs using actual source images and fill a copy of the JSON review template with
Boolean `same_capture`, reviewer ID and reason. Do not mechanically mark every pair false.
Set `NEAR_REVIEW_HINT` and rerun from a fresh kernel with the same input bundles. If coverage
is incomplete, increase the comparison budget or record the remaining limitation explicitly.
Do not alter identifiers, source manifests or test labels to force a preferred allocation.

Save the complete frozen bundle. Future notebooks must verify its fingerprint and content
hashes, load the selected protocol explicitly, select checkpoints/thresholds on development
data, fit calibration only on its reserved data, and leave final tests for a frozen-protocol
evaluation. Any later changes after test inspection must be disclosed. No detector training
is authorized by freezing this data protocol. Full contracts/commands are in `docs/Notebook04.md`.

Stop here. Notebook 05 (data loaders/preprocessing/augmentation) requires a separate instruction
and real audited frozen inputs.